In [1]:
import os
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor


def set_seed(seed: int = 42):
    """Set random seed for reproducibility across numpy, random and python hash seed."""
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)




In [2]:
# Core feature definition (exclude breath_id from model inputs later)
base_feature_cols = ["breath_id", "R", "C", "time_step", "u_in", "u_out"]
target_col = "pressure"
usecols = base_feature_cols + [target_col]

dtypes = {
    "breath_id": np.int32,
    "R": np.int8,
    "C": np.int8,
    "time_step": np.float32,
    "u_in": np.float32,
    "u_out": np.uint8,
    "pressure": np.float32,
}

DATA_ROOT = "/kaggle/input/ventilator-pressure-prediction"
train_path = os.path.join(DATA_ROOT, "train.csv")
df_train = pd.read_csv(train_path, usecols=usecols, dtype=dtypes)

# Feature engineering (kept identical to original pipeline)
df_train["u_in_cum"] = df_train.groupby("breath_id")["u_in"].cumsum().astype(np.float32)
df_train["time_step_sq"] = (df_train["time_step"] ** 2).astype(np.float32)
df_train["u_in_x_u_out"] = (df_train["u_in"] * df_train["u_out"]).astype(np.float32)

df_train["prev_u_in"] = (
    df_train.groupby("breath_id")["u_in"].shift(1).fillna(0).astype(np.float32)
)
df_train["prev_time_step"] = (
    df_train.groupby("breath_id")["time_step"].shift(1).fillna(0).astype(np.float32)
)

df_train["u_in_diff"] = (df_train["u_in"] - df_train["prev_u_in"]).astype(np.float32)
df_train["time_step_diff"] = (
    df_train["time_step"] - df_train["prev_time_step"]
).astype(np.float32)




In [3]:
# Split breaths into training and validation groups
unique_breaths = df_train["breath_id"].unique()
train_breaths, val_breaths = train_test_split(
    unique_breaths, test_size=0.2, random_state=2021
)

train_mask = df_train["breath_id"].isin(train_breaths)
val_mask = df_train["breath_id"].isin(val_breaths)

# Compute breath‑level statistics **only on training breaths** to avoid leakage
breath_stats_train = (
    df_train.loc[train_mask]
    .groupby("breath_id")[target_col]
    .agg(["mean", "std"])
    .rename(columns={"mean": "breath_mean_pressure", "std": "breath_std_pressure"})
    .reset_index()
)
breath_stats_train["breath_mean_pressure"] = breath_stats_train[
    "breath_mean_pressure"
].astype(np.float32)
breath_stats_train["breath_std_pressure"] = breath_stats_train[
    "breath_std_pressure"
].astype(np.float32)

# Merge stats into train and validation sets
df_train = df_train.merge(breath_stats_train, on="breath_id", how="left")
overall_mean = df_train[target_col].mean()
overall_std = df_train[target_col].std()
df_train["breath_mean_pressure"].fillna(overall_mean, inplace=True)
df_train["breath_std_pressure"].fillna(overall_std, inplace=True)

# Define final feature list (excluding the identifier column)
feature_cols = [
    "R",
    "C",
    "time_step",
    "u_in",
    "u_out",
    "u_in_cum",
    "time_step_sq",
    "u_in_x_u_out",
    "breath_mean_pressure",
    "breath_std_pressure",
    "prev_u_in",
    "prev_time_step",
    "u_in_diff",
    "time_step_diff",
]

X_train = df_train.loc[train_mask, feature_cols].values
y_train = df_train.loc[train_mask, target_col].values
X_val = df_train.loc[val_mask, feature_cols].values
y_val = df_train.loc[val_mask, target_col].values

set_seed(2021)



/tmp/ipykernel_11/717365292.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_train["breath_mean_pressure"].fillna(overall_mean, inplace=True)
/tmp/ipykernel_11/717365292.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inpla

In [4]:
# Train a moderately‑sized HistGradientBoostingRegressor
model = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.1,
    max_depth=6,
    loss="absolute_error",
    random_state=2021,
    early_stopping=True,
    n_iter_no_change=20,
)

model.fit(X_train, y_train)

val_pred = model.predict(X_val)
val_mae = mean_absolute_error(y_val, val_pred)
print(f"Validation MAE (raw predictions): {val_mae:.6f}")



Validation MAE (raw predictions): 2.127024


In [5]:
# Prepare test data with identical feature engineering and breath stats
test_path = os.path.join(DATA_ROOT, "test.csv")
test_usecols = base_feature_cols + ["id"]
test_dtypes = {k: dtypes[k] for k in base_feature_cols}
test_dtypes["id"] = np.int32

df_test = pd.read_csv(test_path, usecols=test_usecols, dtype=test_dtypes)

df_test["u_in_cum"] = df_test.groupby("breath_id")["u_in"].cumsum().astype(np.float32)
df_test["time_step_sq"] = (df_test["time_step"] ** 2).astype(np.float32)
df_test["u_in_x_u_out"] = (df_test["u_in"] * df_test["u_out"]).astype(np.float32)

df_test["prev_u_in"] = (
    df_test.groupby("breath_id")["u_in"].shift(1).fillna(0).astype(np.float32)
)
df_test["prev_time_step"] = (
    df_test.groupby("breath_id")["time_step"].shift(1).fillna(0).astype(np.float32)
)

df_test["u_in_diff"] = (df_test["u_in"] - df_test["prev_u_in"]).astype(np.float32)
df_test["time_step_diff"] = (df_test["time_step"] - df_test["prev_time_step"]).astype(
    np.float32
)
df_test = df_test.merge(breath_stats_train, on="breath_id", how="left")
df_test["breath_mean_pressure"].fillna(overall_mean, inplace=True)
df_test["breath_std_pressure"].fillna(overall_std, inplace=True)

X_test = df_test[feature_cols].values
test_pred = model.predict(X_test)

submission = pd.DataFrame(
    {
        "id": df_test["id"],
        "pressure": test_pred.astype(float),
    }
)
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")




/tmp/ipykernel_11/1953426412.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_test["breath_mean_pressure"].fillna(overall_mean, inplace=True)
/tmp/ipykernel_11/1953426412.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inpl

Submission file written to submission.csv


In [6]:
def wc(input_list):
    """Weighted combination of predictions from multiple CSVs."""
    scores = []
    preds = []
    for path in input_list:
        try:
            score = int(path.split("/")[-1].split(".")[1].split(" ")[0])
        except Exception:
            score = 0
        scores.append(score)
        preds.append(pd.read_csv(path).pressure.values)
    if len(preds) == 1:
        return preds[0]
    weight1, weight2 = 0.6, 0.4
    return preds[0] * weight1 + preds[1] * weight2


def blend(a_path, b_path):
    """Blend two submission files."""
    a = pd.read_csv(a_path)
    b = pd.read_csv(b_path)
    a.pressure = a.pressure * 0.7 + b.pressure * 0.3
    a.to_csv("blend.csv", index=False)
    return a